<div style="padding:20px;
            color:white;
            margin:10;
            font-size:200%;
            text-align:center;
            display:fill;
            border-radius:5px;
            background-color:#294B8E;
            overflow:hidden;
            font-weight:700">House price prediction</div>

![](https://cdn.theatlantic.com/thumbor/mD5LrNIJ6KZVeN4bDrD2tCay-3Q=/0x0:2400x1350/960x540/media/img/mt/2022/01/housing_market/original.jpg)

<a id="toc"></a>
- [Introduction](#)
    - [ Import Libraries](#1.1)
    - [Import Data](#1.2)
- [1. Dataset description](#1)
- [2. Exploring data set](#2)
    - [2.1 Correlation](#2.1)
    - [2.2 EDA](#2.1)
- [3. Data preprocessing](#3)
    - [3.1 Pipeline](#3.1)
- [4. Model building](#4)
    - [4.1 Lasso ](#4.1)
    - [4. RandomForest ](#4.1)
- [5. Submitting](#4)

<a id="1"></a>
<div style="padding:20px;
            color:white;
            margin:10;
            font-size:170%;
            text-align:left;
            display:fill;
            border-radius:5px;
            background-color:#294B8E;
            overflow:hidden;
            font-weight:700"><span style='color:#CDA63A'>|</span> Introduction</div>
            
![](https://storage.googleapis.com/kaggle-competitions/kaggle/5407/media/housesbanner.png)            

<a id=""></a>
## <b> <span style='color:#E1B12D'>Import Libraries</span></b> 

In [ ]:
import os
import numpy as np # linear algebra
import pandas as pd # data processing
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_absolute_error,mean_squared_error, mean_squared_log_error)
from xgboost import XGBRegressor, plot_tree
import math

import xgboost as xgb
import hyperopt

pd.set_option("display.max_columns", None)

# Encoders
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from category_encoders.leave_one_out import LeaveOneOutEncoder 
from category_encoders import TargetEncoder, WOEEncoder

# Scalers
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler, RobustScaler, Normalizer

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
import lightgbm as lgb
from sklearn import linear_model
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier,DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures
from xgboost import XGBClassifier
from sklearn.svm import SVC
# manual nested cross-validation for random forest on a classification dataset
from sklearn.datasets import make_classification
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split, StratifiedShuffleSplit,StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import sklearn.metrics as metrics

<a id=""></a>
## <b> <span style='color:#E1B12D'>Import Data</span></b> 

In [ ]:
test=pd.read_csv('/kaggle/input/cleaned-data-house-prices/test_df.csv')
train=pd.read_csv('/kaggle/input/cleaned-data-house-prices/train_df.csv')
sample_subm=pd.read_csv('../input/house-prices-advanced-regression-techniques/sample_submission.csv')

<div style=" background-color:#3b3745;text-align:left; padding: 13px 13px; border-radius: 8px; color: white">
<ul> Due to the large number of NaN values ​​in the real dataset. We load this dataset cleaned. 
    
[Link of dataset](https://www.kaggle.com/datasets/javohirtoshqorgonov/cleaned-data-house-prices)
</ul>
</div>

<a id="1"></a>
<div style="padding:20px;
            color:white;
            margin:10;
            font-size:170%;
            text-align:left;
            display:fill;
            border-radius:5px;
            background-color:#294B8E;
            overflow:hidden;
            font-weight:700">1 <span style='color:#CDA63A'>|</span> Dataset Description</div>

<div style=" background-color:#3b3745;text-align:left; padding: 13px 13px; border-radius: 8px; color: white">
<ul> 
    File descriptions
<li>train.csv - the training set
<li>test.csv - the test set
<li>data_description.txt - full description of each column, originally prepared by Dean De Cock but lightly edited to match the column names used here
<li>sample_submission.csv - a benchmark submission from a linear regression on year and month of sale, lot square footage, and number of bedrooms

    
# Data fields
**Here's a brief version of what you'll find in the data description file.**

<li>SalePrice - the property's sale price in dollars. This is the target variable that you're trying to predict.
<li>MSSubClass: The building class
<li>MSZoning: The general zoning classification
<li>LotFrontage: Linear feet of street connected to property
<li>LotArea: Lot size in square feet
<li>Street: Type of road access
<li>Alley: Type of alley access
<li>LotShape: General shape of property
<li>LandContour: Flatness of the property
<li>Utilities: Type of utilities available
<li>LotConfig: Lot configuration
<li>LandSlope: Slope of property
<li>Neighborhood: Physical locations within Ames city limits
<li>Condition1: Proximity to main road or railroad
<li>Condition2: Proximity to main road or railroad (if a second is present)
<li>BldgType: Type of dwelling
<li>HouseStyle: Style of dwelling
<li>OverallQual: Overall material and finish quality
<li>OverallCond: Overall condition rating
<li>YearBuilt: Original construction date
<li>YearRemodAdd: Remodel date
<li>RoofStyle: Type of roof
<li>RoofMatl: Roof material
<li>Exterior1st: Exterior covering on house
<li>Exterior2nd: Exterior covering on house (if more than one material)
<li>MasVnrType: Masonry veneer type
<li>MasVnrArea: Masonry veneer area in square feet
<li>ExterQual: Exterior material quality
<li>ExterCond: Present condition of the material on the exterior
<li>Foundation: Type of foundation
<li>BsmtQual: Height of the basement
<li>BsmtCond: General condition of the basement
<li>BsmtExposure: Walkout or garden level basement walls
<li>BsmtFinType1: Quality of basement finished area
<li>BsmtFinSF1: Type 1 finished square feet
<li>BsmtFinType2: Quality of second finished area (if present)
<li>BsmtFinSF2: Type 2 finished square feet
<li>BsmtUnfSF: Unfinished square feet of basement area
<li>TotalBsmtSF: Total square feet of basement area
<li>Heating: Type of heating
<li>HeatingQC: Heating quality and condition
<li>CentralAir: Central air conditioning
<li>Electrical: Electrical system
<li>1stFlrSF: First Floor square feet
<li>2ndFlrSF: Second floor square feet
<li>LowQualFinSF: Low quality finished square feet (all floors)
<li>GrLivArea: Above grade (ground) living area square feet
<li>BsmtFullBath: Basement full bathrooms
<li>BsmtHalfBath: Basement half bathrooms
<li>FullBath: Full bathrooms above grade
<li>HalfBath: Half baths above grade
<li>Bedroom: Number of bedrooms above basement level
<li>Kitchen: Number of kitchens
<li>KitchenQual: Kitchen quality
<li>TotRmsAbvGrd: Total rooms above grade (does not include bathrooms)
<li>Functional: Home functionality rating
<li>Fireplaces: Number of fireplaces
<li>FireplaceQu: Fireplace quality
<li>GarageType: Garage location
<li>GarageYrBlt: Year garage was built
<li>GarageFinish: Interior finish of the garage
<li>GarageCars: Size of garage in car capacity
<li>GarageArea: Size of garage in square feet
<li>GarageQual: Garage quality
<li>GarageCond: Garage condition
<li>PavedDrive: Paved driveway
<li>WoodDeckSF: Wood deck area in square feet
<li>OpenPorchSF: Open porch area in square feet
<li>EnclosedPorch: Enclosed porch area in square feet
<li>3SsnPorch: Three season porch area in square feet
<li>ScreenPorch: Screen porch area in square feet
<li>PoolArea: Pool area in square feet
<li>PoolQC: Pool quality
<li>Fence: Fence quality
<li>MiscFeature: Miscellaneous feature not covered in other categories
<li>MiscVal: Value of miscellaneous feature
<li>MoSold: Month Sold
<li>YrSold: Year Sold
<li>SaleType: Type of sale
<li>SaleCondition: Condition of sale
</ul>
</div>

In [ ]:
train.head()

In [ ]:
test.head()

<a id="2"></a>
<div style="padding:20px;
            color:white;
            margin:10;
            font-size:170%;
            text-align:left;
            display:fill;
            border-radius:5px;
            background-color:#294B8E;
            overflow:hidden;
            font-weight:700">2 <span style='color:#CDA63A'>|</span> Exploring data set</div>

In [ ]:
train.info()

In [ ]:
train.describe(include=['O'])

In [ ]:
train.describe()

<a id="2.1"></a>
## <b>2.1 <span style='color:#E1B12D'>Correlation</span></b> 

In [ ]:
train.corr()

In [ ]:
plt.figure(figsize=(30,20))
sns.heatmap(train.corr(),cmap='coolwarm')
plt.show()

In [ ]:
train.corrwith(train['SalePrice']).abs().sort_values(ascending=False)

<div style=" background-color:#3b3745;text-align:left; padding: 13px 13px; border-radius: 8px; color: white">
<ul> 
 The biggest impact is on the price of the house:
<li>OverallQual      0.790982
<li>GrLivArea        0.708624
<li>GarageCars       0.640409
<li>GarageArea       0.623431
<li>TotalBsmtSF      0.613581
<li>1stFlrSF         0.605852
<li>FullBath         0.560664
<li>TotRmsAbvGrd     0.533723
<li>YearBuilt        0.522897
<li>YearRemodAdd     0.507101
<li>GarageYrBlt      0.486362
<li>MasVnrArea       0.477493    
</ul>
</div>

<a id="2.2"></a>
## <b>2.2 <span style='color:#E1B12D'>EDA</span></b> 

In [ ]:
sns.distplot(train['SalePrice'])
plt.show()

In [ ]:
fig,ax=plt.subplots(2,2,figsize=(16,8))
sns.scatterplot(ax=ax[0,0],data=train,y='SalePrice',x='OverallQual')
sns.scatterplot(ax=ax[0,1],data=train,y='SalePrice',x='GrLivArea')
sns.scatterplot(ax=ax[1,0],data=train,y='SalePrice',x='GarageCars')
sns.scatterplot(ax=ax[1,1],data=train,y='SalePrice',x='TotalBsmtSF')

In [ ]:
fig,ax=plt.subplots(2,2,figsize=(16,8))
sns.scatterplot(ax=ax[0,0],data=train,y='SalePrice',x='GarageArea')
sns.scatterplot(ax=ax[0,1],data=train,y='SalePrice',x='1stFlrSF')
sns.scatterplot(ax=ax[1,0],data=train,y='SalePrice',x='FullBath')
sns.scatterplot(ax=ax[1,1],data=train,y='SalePrice',x='TotRmsAbvGrd')

In [ ]:
fig,ax=plt.subplots(2,2,figsize=(16,8))
sns.scatterplot(ax=ax[0,0],data=train,y='SalePrice',x='YearBuilt')
sns.scatterplot(ax=ax[0,1],data=train,y='SalePrice',x='YearRemodAdd')
sns.scatterplot(ax=ax[1,0],data=train,y='SalePrice',x='GarageYrBlt')
sns.scatterplot(ax=ax[1,1],data=train,y='SalePrice',x='MasVnrArea')

In [ ]:
plt.figure(figsize=(16,8))
sns.scatterplot(data=train,x='GrLivArea',y='SalePrice',hue='MSZoning')
plt.show()

In [ ]:
plt.figure(figsize=(16,8))
sns.scatterplot(data=train,x='GarageArea',y='SalePrice',hue='GarageFinish')
plt.show()

In [ ]:
train.columns

<a id="3"></a>
<div style="padding:20px;
            color:white;
            margin:10;
            font-size:170%;
            text-align:left;
            display:fill;
            border-radius:5px;
            background-color:#294B8E;
            overflow:hidden;
            font-weight:700">3 <span style='color:#CDA63A'>|</span> Data preprocessing</div>

In [ ]:
train.drop(['Id'], axis=1, inplace=True)
test.drop(['Id'], axis=1, inplace=True)
train.drop(['Unnamed: 0'], axis=1, inplace=True)
test.drop(['Unnamed: 0'], axis=1, inplace=True)

In [ ]:
train

In [ ]:
test

In [ ]:
train.isnull().sum()[:30]

In [ ]:
train.info()

In [ ]:
train.fillna(method='ffill',inplace=True)
test.fillna(method='ffill',inplace=True)

In [ ]:
X = train.copy()
y = train.SalePrice
X = X.drop('SalePrice',axis=1)

In [ ]:
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [ ]:
cat_cols

<a id="3.1"></a>
> > ## <b> 3.1 <span style='color:#E1B12D'>Pipeline</span></b> 

In [ ]:
from sklearn.impute import SimpleImputer
full_pip=ColumnTransformer([
    ('num',StandardScaler(),num_cols),
    ('cat',OrdinalEncoder(),cat_cols)
    
])

In [ ]:
x=full_pip.fit_transform(X)
test_x=full_pip.fit_transform(test)

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(x,y,test_size=0.2,random_state=4)

In [ ]:
x_train[:2]

In [ ]:
test_x[:2]

<a id="4"></a>
<div style="padding:20px;
            color:white;
            margin:10;
            font-size:170%;
            text-align:left;
            display:fill;
            border-radius:5px;
            background-color:#294B8E;
            overflow:hidden;
            font-weight:700">4 <span style='color:#CDA63A'>|</span>Model building</div>

<a id="4.1"></a>
## <b>4.1 <span style='color:#E1B12D'>Lasso</span></b> 

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.datasets import make_regression
from sklearn.metrics import mean_squared_error,mean_absolute_error

model = LassoCV(cv=10, random_state=19).fit(x_train, y_train)
y_pred_val=model.predict(x_val)
rmse=np.sqrt(mean_squared_error(y_pred_val, y_val))
rmse

<a id="4.2"></a>
## <b>4.2 <span style='color:#E1B12D'>RandomForestRegressor</span></b> 

In [ ]:
rd_model = RandomForestRegressor(n_estimators = 125,max_depth=14, random_state=5)
rd_model.fit(x_train,y_train)

In [ ]:
mse = mean_squared_error(y_val,rd_model.predict(x_val))
rmse = mse**.5
rmse

In [ ]:
pred=rd_model.predict(test_x)

<a id="5"></a>
<div style="padding:20px;
            color:white;
            margin:10;
            font-size:170%;
            text-align:left;
            display:fill;
            border-radius:5px;
            background-color:#294B8E;
            overflow:hidden;
            font-weight:700">5 <span style='color:#CDA63A'>|</span>Submitting</div>

In [ ]:
sample_subm['SalePrice']=pred
sample_subm

In [ ]:
sample_subm.to_csv('subm.csv',index=False)